# SVM在信号分类与质量检测中的应用

## 1. 项目背景

欢迎来到这个机器学习实训，我们将使用红酒数据集，通过构建和训练分类模型来预测红酒的质量。  
红酒的质量评估是一个复杂的过程，通常依赖于专业的品鉴师对多种理化属性（如酸度、糖分、酒精含量等）的感知。  
然而，这种人工评估往往耗时且具有主观性。  
在这个实训中，我们将探索如何利用机器学习技术，根据红酒的理化成分数据来自动预测其质量等级。  
本教程旨在帮助你理解数据预处理、特征工程、SVM训练和评估的整个流程。

**目标**
- 学习如何加载和初步探索数据集。
- 理解数据清洗和特征转换的重要性。
- 掌握使用支持向量机 SVM 进行分类。
- 学会评估分类模型的性能。

## 2. 数据处理

在进行数据分析和机器学习任务时，我们首先需要导入一些 Python 库。  
这些库提供了处理数据、构建模型、进行数据可视化等所需的功能。

In [ ]:
# 数据处理和分析库
import pandas as pd  # 用于数据结构（如DataFrame）和数据分析工具
import numpy as np   # 用于科学计算，特别是数组操作

# 数据可视化库
import matplotlib.pyplot as plt # 用于创建静态、动态、交互式的可视化图表
import seaborn as sns           # 基于matplotlib的统计图表库，提供更高级的接口和美观的默认样式

# 机器学习模型和工具
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # 用于评估分类模型的性能
from sklearn.metrics import precision_score, recall_score, f1_score               # 评估分类的精确率、召回率和F1分数
from sklearn.model_selection import train_test_split                          # 用于将数据集分割成训练集和测试集
from sklearn.preprocessing import StandardScaler                              # 数据预处理：特征标准化
from sklearn.svm import SVC                                                   # 支持向量机分类器 (Support Vector Classifier)

本实训将使用 UCI 机器学习库中的[红酒质量数据集](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)。  

### 2.1 加载数据集

加载数据集后，我们通常会查看前几行数据，以快速了解数据的结构和内容。

In [ ]:
dataset_path = '/home/jovyan/work/datasets/68872f804e03dbf50518a08a-momodel/winequality-red.csv'
# 使用 pandas 的 read_csv 函数加载 CSV 文件到 DataFrame
# 确保 'dataset/winequality-red.csv' 路径与你的文件实际位置匹配
df = pd.read_csv(dataset_path)

# 显示 DataFrame 的前5行数据，帮助我们快速了解数据集的结构和内容
df.head()

df.head()会显示数据集的前5行数据。  
通过这个输出，我们可以看到数据集包含多列，每列代表红酒的一个理化属性。  
- fixed acidity (固定酸度): 主要指酒中不挥发性酸的含量。  
- volatile acidity (挥发性酸度): 主要指酒中可挥发性酸（主要是乙酸）的含量，过高会导致醋味。
- citric acid (柠檬酸): 少量存在于葡萄酒中，能增加葡萄酒的“新鲜度”和风味。
- residual sugar (残糖): 发酵结束后葡萄酒中剩余的糖分含量，影响甜度。
- chlorides (氯化物): 酒中氯化钠（食盐）的含量，影响口感。
- free sulfur dioxide (游离二氧化硫): 活性形式的二氧化硫，用于防止葡萄酒氧化和细菌生长。
- total sulfur dioxide (总二氧化硫): 葡萄酒中游离和结合二氧化硫的总量。
- density (密度): 葡萄酒的密度，与酒精和糖的含量密切相关。
- pH (酸碱度): 衡量葡萄酒的酸度或碱度，影响葡萄酒的颜色、稳定性和口感。
- sulphates (硫酸盐): 指葡萄酒中硫酸钾等硫酸盐的含量，通常作为抗菌剂和抗氧化剂添加。
- alcohol (酒精): 葡萄酒中酒精的含量，影响酒体、口感和香气。
- quality (质量): 目标变量，通常是0-10的评分，表示红酒的质量（这是我们要预测的）。

### 2.2 数据探索

在开始任何模型构建之前，深入理解数据集是至关重要的一步。  
这包括检查数据的基本信息、数据类型、统计摘要、缺失值和重复值等。  
通过这些探索性数据分析 (EDA) 步骤，我们可以发现数据中的潜在问题，为后续的数据清洗和预处理做准备。

**学习目标**  

- 学会检查数据集的维度、数据类型和基本统计信息。  
- 掌握如何识别和处理缺失值及重复值。  
- 通过可视化方法探索数据分布和特征间的关系。  

In [ ]:
# 获取数据集的形状（行数和列数）
# 输出格式为 (行数, 列数)
df.shape

表示这个数据集有 1599 行（即 1599 个红酒样本）和 12 列（即 11 个特征变量和 1 个目标变量`quality`）。

In [ ]:
# 获取数据集的摘要信息，包括每列的非空值数量、数据类型和内存使用情况
df.info()

`df.info()`提供了每列的非空值数量和数据类型。  
从输出中可以看到，所有列的“Non-Null Count”都是1599，这说明数据集中没有缺失值。  
所有特征都是浮点型 (`float64`)，而目标变量`quality`是整型 (`int64`)。  
这对于后续的数据预处理和模型选择是重要的信息。

In [ ]:
# 生成数据集中数值型列的描述性统计信息
# 包括计数、均值、标准差、最小值、25%分位数、中位数（50%分位数）、75%分位数和最大值
df.describe()

`df.describe()`的输出提供了每列的统计摘要。  
通过这些统计量，我们可以初步了解数据的分布、范围和潜在的异常值。  
例如，`min`和`max`值可以帮助我们了解数据的边界，`mean`和`std`可以告诉我们数据的集中趋势和离散程度。  
观察各列的均值和中位数，如果差异较大，可能表明数据分布偏斜。

In [ ]:
# 检查数据集中每列的缺失值数量
# .isna() 会返回一个布尔型DataFrame，其中True表示缺失值
# .sum() 会对每列的True值进行求和，从而得到缺失值的总数
df.isna().sum()

**思考：为什么检查缺失值很重要？如果存在缺失值应该如何处理？**  
缺失值会影响模型的训练，导致模型性能下降或无法正常运行。如果存在缺失值，常见的处理方法包括：  
- 删除：删除包含缺失值的行或列（如果缺失值数量很少或该列不重要）。
- 填充：
  - 均值/中位数/众数填充：用该列的统计量填充缺失值。适用于数值型或类别型数据。
  - 预测填充：使用其他特征来预测缺失值，例如通过回归模型。
  - 特定值填充：用0、-1或其他有意义的值填充。

在这个数据集中，`df.isna().sum()`的输出显示所有列的缺失值数量都为 0，这意味着数据非常干净，无需进行缺失值处理。

In [ ]:
# 检查数据集中是否存在重复行，并计算重复行的数量
# .duplicated() 会返回一个布尔型Series，其中True表示该行是重复的（除了第一次出现）
# .sum() 会对Series中的True值进行求和，得到重复行的总数
df.duplicated().sum()

**思考：为什么需要检查和处理重复数据？如何处理？**  
重复数据可能导致模型对某些样本的过度学习，从而影响模型的泛化能力。  
如果存在重复行，通常会选择删除重复的行以确保数据集的唯一性。
`df.duplicated().sum()`的输出显示有 240 个重复行。  
这表明数据集中存在完全相同的样本，我们应该在特征工程或模型训练之前将其删除。

### 2.3 数据可视化

特征分布图（直方图）

In [ ]:
# 可视化每个特征的分布情况，以判断是否存在偏斜或异常值
# 设置图表的布局，创建 3x4 的子图网格
fig, axs = plt.subplots(3, 4, figsize=(20, 15))

# 展平 axs 数组以便于迭代
axs = axs.flatten()

# 遍历 DataFrame 的所有列，为每一列绘制直方图
# df.columns[:-1] 表示除了最后一列（'quality'）之外的所有特征列
for i, col in enumerate(df.columns[:-1]):
    sns.histplot(df[col], kde=True, ax=axs[i]) # 绘制直方图，kde=True 表示同时绘制核密度估计曲线
    axs[i].set_title(f'Distribution of {col}') # 设置子图标题

# 删除多余的空子图（如果特征数量不是 3x4 的整数倍）
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

# 调整布局，确保所有子图和标题不重叠
plt.tight_layout()

# 显示图表
plt.show()

我们可以观察到一些特征（如`fixed acidity, volatile acidity, citric acid, chlorides, free sulfur dioxide, total sulfur dioxide, sulphates`）可能存在偏斜分布。  
这意味着数据在某个方向上集中，并且尾部较长。  
对于偏斜的数据，可能需要进行特征转换（如对数变换、平方根变换）来使其更接近正态分布，这有助于一些机器学习模型的性能。

`residual sugar` 和 `alcohol` 看起来也有些偏斜，但可能程度稍轻。

`pH` 和 `density` 的分布相对更接近正态。

In [ ]:
# 绘制目标变量 'quality' 的计数图，查看其分布
plt.figure(figsize=(8, 6)) # 设置图表大小
sns.countplot(x='quality', data=df) # 绘制计数图
plt.title('Distribution of Red Wine Quality') # 设置图表标题
plt.xlabel('Quality (0-10)') # 设置 x 轴标签
plt.ylabel('Count') # 设置 y 轴标签
plt.show()

质量分布图（计数图）  

我们可以看到红酒质量的评分主要集中在5、6和7分。  
这表明数据集中存在类别不平衡问题，即某些质量等级的样本数量远多于其他等级。  
例如，质量为 3 和 8 的样本数量非常少。

挑战与解决方案：类别不平衡是分类任务中常见的问题。  
它可能导致模型在预测少数类别时表现不佳。  
应对方法包括：

- 过采样 (Oversampling)：增加少数类别的样本数量（如 SMOTE）。
- 欠采样 (Undersampling)：减少多数类别的样本数量。
- 使用不同的评估指标：除了准确率，还应关注 F1-score、精确率、召回率、混淆矩阵等，它们能更好地反映模型在不平衡数据集上的表现。
- 调整模型参数：有些模型允许设置类别权重。

In [ ]:

# 计算各个特征与质量之间的相关性，并可视化为热力图
# .corr() 计算所有列之间的相关系数
# .abs() 取相关系数的绝对值，关注相关强度而不考虑方向
plt.figure(figsize=(12, 10)) # 设置图表大小
sns.heatmap(df.corr().abs(), annot=True, cmap='coolwarm', fmt=".2f") # 绘制热力图，显示相关系数值，使用颜色映射
plt.title('Correlation Matrix of Red Wine Features') # 设置图表标题
plt.show()

相关性热力图  

热力图显示了各个特征之间以及特征与目标变量`quality`之间的相关性。

与`quality`相关性较高的特征：  
`alcohol`（正相关）和 `volatile acidity`（负相关）与红酒质量的相关性最高。  
这表明酒精度越高，挥发性酸度越低，红酒质量可能越好。  
`sulphates`和`citric acid`也呈现正相关。

特征间的相关性：  
有些特征之间也存在较高的相关性，例如 `fixed acidity`与`citric acid、density`与`fixed acidity`和`alcohol`之间。  
高相关性特征可能导致多重共线性问题，影响模型的解释性，但对于像 SVM 这样的模型，通常不是主要问题。

### 2.4 数据预处理

数据预处理是机器学习流程中至关重要的一步，它将原始数据转换为适合模型训练的格式。  
这包括处理重复值、对目标变量进行分类转换（如果需要）以及特征缩放。  

**学习目标**  

- 掌握如何删除重复数据。  
- 理解为什么需要将连续的质量评分转换为二分类或多分类标签。  
- 学会使用`StandardScaler`进行特征标准化，并理解其重要性。
- 理解训练集和测试集划分的意义。

#### 2.4.1 处理重复数据

根据之前的数据探索，我们发现数据集中存在重复行。  
为了避免模型对重复样本的过度学习，我们应该删除这些重复数据。

In [ ]:
# 删除DataFrame中的重复行
# drop_duplicates() 默认保留第一次出现的行，删除所有后续的重复行
df.drop_duplicates(inplace=True)

# 再次检查重复行数量，确认是否已成功删除
df.duplicated().sum()

删除重复行后，数据集的行数会减少，因为重复的样本被移除了。  
`inplace=True`参数表示直接在原DataFrame上修改，而不是返回一个新的DataFrame。

#### 2.4.2 目标变量转换

原始数据集的`quality`列是 0-10 之间的连续整数评分。  
对于分类任务，我们通常需要将其转换为离散的类别标签。  
在这个实训中，我们将把红酒质量分为“好”和“不好”两类，这在实际应用中更常见且更具操作性。  

分类策略：
- 我们设定一个阈值（例如，质量评分大于或等于 7 的视为“好酒”，否则为“不好的酒”）。
- 将原始的 0-10 评分转换为二分类标签 (0 或 1)。

In [ ]:
# 定义一个函数，用于将红酒质量评分转换为二分类标签
# 如果 quality >= 7，则返回 1 (代表“好酒”)
# 否则返回 0 (代表“不好的酒”)
def classify_quality(quality):
    if quality >= 7:
        return 1  # 1 代表 '好酒'
    else:
        return 0  # 0 代表 '不好的酒'

# 将这个函数应用到 'quality' 列，创建新的目标变量 'quality_category'
# 使用 .apply() 方法将函数逐行应用到指定列
df['quality_category'] = df['quality'].apply(classify_quality)

# 再次查看新的目标变量 'quality_category' 的分布
# sns.countplot 可以很直观地展示每个类别的样本数量
plt.figure(figsize=(6, 4))
sns.countplot(x='quality_category', data=df)
plt.title('Distribution of Red Wine Quality Categories')
plt.xlabel('Quality Category (0: Bad, 1: Good)')
plt.ylabel('Count')
plt.show()

# 确认原始 'quality' 列是否可以删除
# 打印新的DataFrame头部，观察 'quality_category' 列
df.head()

**思考：为什么需要将连续的质量评分转换为二分类标签？这种转换的优缺点是什么？**
- 原因：
  - 简化问题：将多类别或连续值问题转换为二分类，可以简化模型任务，尤其当原始类别分布不均衡时。
  - 实际应用：在很多业务场景中，我们可能更关心产品是“合格”还是“不合格”，“好”还是“差”，而不是精确的细分等级。
  - 机器学习模型：某些分类模型更适合处理二分类问题。

- 优点：
  - 模型训练更简单：减少了分类器的复杂性。
  - 结果更直观：更容易解释和理解。

- 缺点：

  - 信息损失：将多个质量等级合并为两个，会丢失原始评分中的细粒度信息。例如，质量为5和6的红酒都被标记为“不好”，但它们之间仍有区别。
  - 阈值选择：阈值的选择是主观的，不同的阈值可能导致不同的分类结果和模型性能。

#### 2.4.3 特征与目标分离

在训练机器学习模型之前，我们需要将数据集分为特征 (X) 和目标变量 (y)。  
特征是用于预测的输入，目标变量是我们希望模型预测的输出。

In [ ]:
# 定义特征变量 X：包含除 'quality' 和 'quality_category' 之外的所有列
# .drop() 方法用于删除指定的列
# axis=1 表示删除列，而不是行
X = df.drop(['quality', 'quality_category'], axis=1)

# 定义目标变量 y：我们新创建的二分类质量类别 'quality_category'
y = df['quality_category']

# 打印 X 和 y 的前5行，验证分离是否正确
print("Features (X) head:")
print(X.head())
print("\nTarget (y) head:")
print(y.head())

#### 2.4.4 数据标准化

在许多机器学习算法中，对特征进行缩放是必不可少的一步，尤其对于像SVM等基于距离的算法。  
数据标准化（Standardization）通过将特征转换为均值为 0、标准差为 1 的分布，可以消除特征量纲对模型训练的影响。  

**为什么需要标准化？**

- 算法敏感性：某些算法对特征的尺度非常敏感。例如，如果一个特征的取值范围远大于另一个特征，那么在计算距离时，取值范围大的特征会占据主导地位，使得模型偏向于该特征。
- 加快收敛：对于梯度下降等优化算法，标准化可以使损失函数更平滑，从而加快模型的收敛速度。
- 提高性能：通过标准化，可以确保所有特征对模型的贡献是公平的，从而提高模型的整体性能。

In [ ]:
# 初始化 StandardScaler 对象
# StandardScaler 会对数据进行 z-score 标准化：(X - mean) / std_dev
scaler = StandardScaler()

# 对特征数据 X 进行标准化
# .fit_transform() 首先计算训练数据的均值和标准差（fit），然后用这些值对数据进行标准化（transform）
# 注意：Scaler 应该只在训练数据上 fit，然后用同样的 scaler 来 transform 训练数据和测试数据
X_scaled = scaler.fit_transform(X)

# 将标准化后的 NumPy 数组转换回 Pandas DataFrame，并保留原始列名
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 打印标准化后数据的前5行，观察数值变化
print("Scaled Features (X_scaled) head:")
print(X_scaled.head())

**思考：标准化后的数据与原始数据相比有什么不同？为什么我们通常只在训练集上fit标准化器，然后用它来transform训练集和测试集？**  

标准化后的数据每列的均值接近0，标准差接近1。数值范围也发生了变化，不再是原始的物理单位。

**只在训练集上fit标准化器并应用于测试集的原因：**  
这是为了避免数据泄露 (Data Leakage)。如果在整个数据集上fit标准化器（包括测试集），  
那么测试集的信息就会“泄露”到训练过程中，导致模型在测试集上的表现看起来很好，但在实际新数据上表现不佳。

正确的做法是：
- 在训练集上fit标准化器，学习训练集的均值和标准差。
- 用这个已经fit好的标准化器去transform训练集。
- 用同一个已经fit好的标准化器去transform测试集。

#### 2.4.5 划分训练集和测试集

为了评估模型在未见过数据上的泛化能力，我们需要将数据集划分为训练集和测试集。  
训练集用于模型的学习，测试集用于评估模型性能。

In [ ]:
# 将数据划分为训练集和测试集
# X_scaled: 标准化后的特征数据
# y: 目标变量（红酒质量类别）
# test_size=0.2: 20% 的数据作为测试集，80% 作为训练集
# random_state=42: 确保每次运行时数据集划分的结果一致，便于复现
# stratify=y: 确保训练集和测试集中目标变量 (y) 的类别比例与原始数据集保持一致，这对于类别不平衡的数据集尤其重要
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 打印划分后数据集的形状，进行确认
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# 检查训练集和测试集中目标变量的类别分布，确保 stratify 参数生效
print("\nDistribution of y_train:")
print(y_train.value_counts(normalize=True)) # normalize=True 显示比例
print("\nDistribution of y_test:")
print(y_test.value_counts(normalize=True))

**思考：为什么需要划分训练集和测试集？`random_state`和`stratify`参数的作用是什么？**

划分目的：
- 评估泛化能力：模型在训练集上学习，但在测试集上评估其在未知数据上的表现。这能有效避免过拟合。
- 避免数据泄露：确保测试集是模型从未见过的数据。

random_state：
- 这是一个随机种子。设置相同的random_state值，可以保证每次运行`train_test_split`时，数据的划分结果都是相同的。  
  这对于实验结果的可复现性非常重要。

stratify=y：

- 当数据集中的类别不平衡时（例如，“好酒”样本远少于“不好酒”样本），`stratify=y`会确保训练集和测试集中，各个类别的样本比例与原始数据集中的比例保持一致。  
  这有助于模型在训练时看到所有类别的代表性样本，并在测试时对其进行公平评估。

## 3. 模型构建与训练

现在我们已经完成了数据预处理，可以开始构建和训练机器学习模型了。  
本实训将使用支持向量机 (Support Vector Machine)实现分类功能。  

学习目标：  
- 学习如何初始化、训练和使用支持向量机模型。
- 理解这些模型的基本原理和适用场景。  

SVM 是一种强大的二分类模型，它通过找到一个最优超平面来最大化不同类别数据点之间的间隔。  
当数据线性不可分时，SVM 可以通过核函数（Kernel Function）将数据映射到高维空间，从而实现非线性分类。

In [ ]:
# 初始化支持向量机分类器
# kernel='rbf': 使用径向基函数 (Radial Basis Function) 核，也称为高斯核。
#                RBF核在处理非线性可分数据时非常有效，是SVM中最常用的核函数之一。
#                它通过将数据映射到无限维空间，使得数据在该高维空间中变得可分。
# random_state=42: 确保每次运行结果一致的随机种子
svm_model = SVC(kernel='rbf', random_state=42)

# 使用训练数据训练 SVM 模型
svm_model.fit(X_train, y_train)

# 使用训练好的模型对测试集进行预测
y_pred_svm = svm_model.predict(X_test)

**思考：为什么 SVM 特别适合信号分类和质量检测？RBF 核函数的作用是什么？**

- SVM的优势：
  - 处理高维数据：信号数据经过特征提取后，维度通常较高，SVM 在处理高维数据时表现良好。
  - 处理非线性可分问题：信号特征空间中的类别边界往往是非线性的，RBF 核函数能有效处理这种非线性关系。
  - 全局最优解：SVM 的优化目标是凸二次规划问题，可以找到全局最优解，而不是像神经网络那样容易陷入局部最优。
  - 鲁棒性：通过最大化间隔，SVM 对训练数据中的噪声和异常值具有一定的鲁棒性。

- RBF 核函数 (Radial Basis Function Kernel)：
  - RBF核是一种非线性核函数。它的核心思想是将原始特征空间中的数据点映射到一个更高维的特征空间，  
    在这个新的高维空间中，原来线性不可分的数据可能变得线性可分。

  - 数学上，RBF 核的计算依赖于数据点之间的欧氏距离，越近的数据点，其核函数值越大。

  - 在处理非线性可分的数据集时，RBF 核是最常用且通常效果最好的核函数。  
    它能够捕捉数据中复杂的非线性关系，使得 SVM 能够构建复杂的决策边界。

## 4. 模型评估与比较

### 4.1 计算评估指标


我们将计算每个模型的准确率 (Accuracy)、精确率 (Precision)、召回率 (Recall) 和 F1-score。

- 准确率 (Accuracy)：所有预测正确的样本占总样本的比例。  
  - accuracy_score(y_true, y_pred)  
  
- 精确率 (Precision)：在所有被预测为正类别的样本中，真正是正类别的比例。
  - precision_score(y_true, y_pred, average='weighted')

- 召回率 (Recall)：在所有真正是正类别的样本中，被模型正确预测为正类别的比例。
  - recall_score(y_true, y_pred, average='weighted')

- F1-score：精确率和召回率的调和平均值，是衡量模型综合性能的指标，尤其在类别不平衡时比准确率更具参考价值。
  - f1_score(y_true, y_pred, average='weighted')

对于多分类或二分类但希望考虑所有类别的情况，`average='weighted'` 会在计算时考虑每个类别的样本数量进行加权平均。

In [ ]:
# 定义模型名称列表
models = ['SVM']

# 定义预测结果列表
y_pred_list = [y_pred_svm]

# 初始化列表来存储模型的评估指标
accuracy = []
precision = []
recall = []
f1 = []

# 遍历模型的预测结果，计算并存储评估指标
for i, y_pred in enumerate(y_pred_list):
    # 计算准确率
    accuracy.append(accuracy_score(y_test, y_pred))
    # 计算精确率，使用 'weighted' 平均，考虑类别不平衡
    precision.append(precision_score(y_test, y_pred, average='weighted'))
    # 计算召回率，使用 'weighted' 平均
    recall.append(recall_score(y_test, y_pred, average='weighted'))
    # 计算 F1-score，使用 'weighted' 平均
    f1.append(f1_score(y_test, y_pred, average='weighted'))

# 打印模型的评估指标
print("Model Evaluation Metrics:")
for i, model_name in enumerate(models):
    print(f"\n--- {model_name} ---")
    print(f"Accuracy: {accuracy[i]:.4f}")
    print(f"Precision: {precision[i]:.4f}")
    print(f"Recall: {recall[i]:.4f}")
    print(f"F1-score: {f1[i]:.4f}")

# 绘制柱状图比较模型的性能指标
fig, axs = plt.subplots(2, 2, figsize=(12, 8)) # 创建 2x2 的子图布局

# 定义一个字典，将指标名称映射到其对应的数值列表
metrics = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-score': f1}

# 遍历每个指标，并在相应的子图中绘制柱状图
for i, (metric_name, metric_values) in enumerate(metrics.items()):
    ax = axs[i // 2, i % 2] # 计算当前指标对应的子图位置
    bars = ax.bar(models, metric_values, color=['blue', 'green', 'red']) # 绘制柱状图
    ax.set_title(metric_name) # 设置子图标题
    ax.set_ylabel(metric_name) # 设置 y 轴标签
    ax.set_ylim(0.6, .9) # 设置 y 轴的显示范围，使图表更清晰
    
    # 在每个柱状图上方显示具体的数值
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 4), va='bottom', ha='center') # 在柱状图上方居中显示数值

# 调整布局，防止重叠
plt.tight_layout()

# 保存图表到 docs 文件夹
# plt.savefig("docs/metrics_comparison.png")

# 显示图表
plt.show()

### 4.2 混淆矩阵 (Confusion Matrix)

混淆矩阵是一个表格，用于可视化分类算法的性能。  
它显示了真阳性 (TP)、真阴性 (TN)、假阳性 (FP) 和假阴性 (FN) 的数量，从而更详细地揭示模型在每个类别上的表现。

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay # 导入混淆矩阵可视化工具

# 定义类别标签
class_labels = ['Bad Quality (0)', 'Good Quality (1)']

# 为模型绘制混淆矩阵
fig, axs = plt.subplots(1, 1, figsize=(6, 6)) # 创建 1x3 的子图布局

for i, model_name in enumerate(models):
    ax = axs
    # 使用 ConfusionMatrixDisplay.from_estimator 或 from_predictions 绘制混淆矩阵
    # 这里我们使用 from_predictions 因为我们已经有了 y_test 和 y_pred
    # display_labels 参数用于设置类别标签的显示名称
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_list[i],
                                            display_labels=class_labels,
                                            cmap=plt.cm.Blues,
                                            ax=ax)
    ax.set_title(f'{model_name} Confusion Matrix') # 设置子图标题

plt.tight_layout()
plt.show()

**思考：如何从混淆矩阵中解读模型的性能？为什么混淆矩阵在类别不平衡时特别重要？** 

- 混淆矩阵解读：
  - 行代表真实类别，列代表预测类别。
  - 主对角线上的值（从左上到右下）代表正确分类的样本数。
  - 非主对角线上的值代表错误分类的样本数。
    - 假阳性 (FP)：模型预测为正类，但实际是负类（例如，把“不好酒”预测为“好酒”）。
    - 假阴性 (FN)：模型预测为负类，但实际是正类（例如，把“好酒”预测为“不好酒”）。

- 类别不平衡时的重要性：
  - 在类别不平衡的数据集中，仅仅依靠准确率可能会产生误导。  
    例如，如果95%的样本是负类，一个总是预测为负类的模型可以达到95%的准确率，但它对正类别的预测能力为零。
  - 混淆矩阵能清楚地展示模型在每个类别上的性能，特别是对少数类别的识别能力（召回率）和预测准确性（精确率）。
  - 通过混淆矩阵，我们可以看到模型是将少数类别错分为多数类别（导致高FN），还是将多数类别错分为少数类别（导致高FP）。  
    这有助于我们针对性地优化模型。

### 4.3 分类报告 (Classification Report)

`classification_report`函数提供了一个文本报告，其中包含了精确率、召回率、 F1-score 以及每个类别的支持度（样本数量）。

In [ ]:
# 为每个模型生成并打印分类报告
print("\n--- Classification Reports ---")
for i, model_name in enumerate(models):
    print(f"\n--- {model_name} Classification Report ---")
    # classification_report 会为每个类别计算精确率、召回率和 F1-score
    # target_names 参数用于设置类别名称的显示
    print(classification_report(y_test, y_pred_list[i], target_names=class_labels))

**思考：分类报告中的 support 和 macro avg/weighted avg 有什么区别？**

- support：表示每个类别在 y_test 中实际的样本数量。
- macro avg (宏平均)：简单地计算每个类别的指标（精确率、召回率、F1-score），然后取这些指标的平均值。  
  它不考虑每个类别的样本数量，因此会平等对待每个类别。在类别不平衡时，macro avg 更能反映模型在少数类别上的表现。

- weighted avg (加权平均)：计算每个类别的指标，然后根据每个类别的 support （样本数量）进行加权平均。  
  这使得多数类别的指标对总平均值的影响更大。weighted avg 的 F1-score 通常与整体准确率更接近。

从上面的可视化和评估指标中，我们可以得出以下结论：

从图中可以看出，支持向量机 (SVM) 在所有评估指标（准确率、精确率、召回率、F1-score）上都表现不错。  

## 5. 总结

通过本次实训，我们学习了从头到尾完成一个机器学习分类任务：从数据加载、探索性数据分析、数据预处理、模型训练和评估。

**实训回顾**

数据加载与理解：我们导入了必要的库，加载了红酒质量数据集，并使用`df.head()`、`df.shape`、`df.info()`、`df.describe()`、  
`df.isna().sum()`和`df.duplicated().sum()`对数据进行了初步探索。  
我们发现数据中存在重复值和目标变量类别不平衡的问题。

数据预处理：我们删除了重复行，并将多分类的红酒质量评分转换为二分类（“好酒”和“不好的酒”），以简化分类任务。  
随后，我们对特征数据进行了标准化处理，以消除量纲影响，并使用`train_test_split`将数据划分为训练集和测试集，确保模型评估的公正性。

模型构建与训练：我们使用支持向量机训练了模型。

模型评估与比较：我们计算了准确率、精确率、召回率和 F1-score 等多种评估指标，并通过柱状图和混淆矩阵直观展现模型性能。

**进一步的探索和学习**  

模型调优：可以尝试对每个模型的超参数进行调优（例如，SVM 的 C 和 gamma 参数等），使用`GridSearchCV`或`RandomizedSearchCV`来找到最优参数组合。

处理类别不平衡：可以尝试使用过采样（如SMOTE）或欠采样技术来平衡数据集中的类别分布，观察是否能进一步提升模型在少数类别上的性能。

特征工程：可以尝试创建新的特征，例如特征组合或多项式特征，看看是否能捕捉到数据中更复杂的模式。

其他模型：尝试其他分类模型，如逻辑回归、梯度提升树（XGBoost, LightGBM）等，并比较它们的性能。

解释性：对于表现最好的模型，尝试进行模型解释性分析，理解哪些特征对红酒质量的分类贡献最大。